# Advanced Python Problems and Solutions: `__slots__` and Single Inheritance

**24 advanced problems · runnable solutions · automated assertions · standard-library only · Python 3.10+**

This notebook expands the supplied teaching material about slotted and ordinary bases, slotted and ordinary subclasses, property backing storage, descriptors, and opting into `__dict__`. Each exercise has a challenge, a detailed worked solution, executable tests and a further practice task. Problems also explore error handling, `weakref`, dataclasses, reflection, serialization, copying, memory measurement and a complete capstone.

**Instructions:** Run **Kernel → Restart & Run All**. Each code cell asserts its own expected behavior, including intended exceptions. Distinct class names make it safe to run the notebook sequentially. All benchmark numbers depend on interpreter and machine; no numerical performance result is asserted. Slotted attributes are not automatically type-checked or immutable.

## Test helper — verify exception *types*, not fragile error-message strings

In [1]:
import sys
from typing import Callable

def expect_raises(exception_type, fn: Callable, *args, **kwargs):
    try:
        fn(*args, **kwargs)
    except exception_type as exc:
        return exc
    except Exception as exc:
        raise AssertionError(f'Expected {exception_type.__name__}, got {type(exc).__name__}') from exc
    raise AssertionError(f'Expected {exception_type.__name__}; nothing was raised')

assert sys.version_info >= (3, 10)
print('Python:', sys.version.split()[0])

Python: 3.13.7


## Problem 01: The unexpected dictionary in an unslotted child

**Challenge:** Build a base with a `name` slot and an unslotted child. Show that `name` remains in the inherited slot, while a new `age` uses the child dictionary.

**Before revealing the solution:** predict which fields are slotted, which enter an instance dictionary, and which operations should raise errors, wherever applicable.

### Worked solution and reasoning

Declaring slots in the base does not force descendants to be slotted. A subclass without `__slots__` normally adds `__dict__`, while its parent-defined slots remain intact.

In [2]:
class P01Base:
    __slots__ = ('name',)
    def __init__(self, name): self.name = name

class P01Child(P01Base):
    pass

base = P01Base('base')
child = P01Child('child')
assert not hasattr(base, '__dict__')
assert child.name == 'child' and child.__dict__ == {}
child.age = 27
assert child.__dict__ == {'age': 27}
assert 'name' not in child.__dict__
print('01:', child.name, vars(child))

01: child {'age': 27}


**Further practice:** Add `child.city` and inspect how `vars(child)` changes.

## Problem 02: Keep a descendant fixed-layout without new slots

**Challenge:** Create a slotted base and a child that declares no new data. The child must inherit the field and reject unknown attributes.

**Before revealing the solution:** predict which fields are slotted, which enter an instance dictionary, and which operations should raise errors, wherever applicable.

### Worked solution and reasoning

An explicit empty slots tuple says the child introduces no instance dictionary and no new slots. It does not remove storage inherited from the base.

In [3]:
class P02Base:
    __slots__ = ('name',)
    def __init__(self, name): self.name = name

class P02Child(P02Base):
    __slots__ = ()

obj = P02Child('Taylor')
assert obj.name == 'Taylor'
assert not hasattr(obj, '__dict__')
expect_raises(AttributeError, setattr, obj, 'age', 20)
assert P02Child.__slots__ == ()
print('02: inherited field works; arbitrary fields forbidden')

02: inherited field works; arbitrary fields forbidden


**Further practice:** Create a grandchild with `__slots__ = ()` and check the same properties.

## Problem 03: Introduce exactly two child slots and initialize via super()

**Challenge:** Give a student inherited `name` and new `school` and `student_number` fields. Validate inputs, call `super()`, and prevent accidental additional fields.

**Before revealing the solution:** predict which fields are slotted, which enter an instance dictionary, and which operations should raise errors, wherever applicable.

### Worked solution and reasoning

Subclass slots contain new fields only. `super()` reuses base validation and ensures the inherited slot gets initialized correctly.

In [4]:
class P03Person:
    __slots__ = ('name',)
    def __init__(self, name):
        if not isinstance(name, str) or not name.strip():
            raise ValueError('name required')
        self.name = name.strip()

class P03Student(P03Person):
    __slots__ = ('school', 'student_number')
    def __init__(self, name, school, student_number):
        super().__init__(name)
        if not isinstance(student_number, str) or not student_number.strip():
            raise ValueError('student number required')
        self.school = school
        self.student_number = student_number

student = P03Student('  Noor ', 'Example School', 'S-1')
assert (student.name, student.school, student.student_number) == ('Noor', 'Example School', 'S-1')
expect_raises(ValueError, P03Student, 'Noor', 'School', '')
expect_raises(AttributeError, setattr, student, 'graduation_year', 2030)
assert not hasattr(student, '__dict__')
print('03:', student.name, student.school, student.student_number)

03: Noor Example School S-1


**Further practice:** Add a slotted grandchild with one new `thesis_topic` slot, chaining `super()` through both ancestors.

## Problem 04: A child cannot eliminate the dictionary inherited from its parent

**Challenge:** Make an ordinary parent with a `name` in its dictionary and a slotted child with `age`. Check where each value lives and add an arbitrary attribute.

**Before revealing the solution:** predict which fields are slotted, which enter an instance dictionary, and which operations should raise errors, wherever applicable.

### Worked solution and reasoning

An inherited instance dictionary survives a child slot declaration; the new slot is separate storage from that dictionary.

In [5]:
class P04Parent:
    def __init__(self, name): self.name = name

class P04Child(P04Parent):
    __slots__ = ('age',)
    def __init__(self, name, age):
        super().__init__(name)
        self.age = age

obj = P04Child('Ada', 29)
assert obj.__dict__ == {'name': 'Ada'}
assert obj.age == 29 and 'age' not in obj.__dict__
obj.favorite = 'math'
assert obj.__dict__ == {'name': 'Ada', 'favorite': 'math'}
print('04:', vars(obj), 'slotted age:', obj.age)

04: {'name': 'Ada', 'favorite': 'math'} slotted age: 29


**Further practice:** What changes if the parent declares `__slots__ = ("name",)`? Test before answering.

## Problem 05: Opt in to hybrid storage intentionally

**Challenge:** Create one class with a slotted ID and open-ended annotations. Allow annotations while keeping the ID out of the instance dictionary.

**Before revealing the solution:** predict which fields are slotted, which enter an instance dictionary, and which operations should raise errors, wherever applicable.

### Worked solution and reasoning

Explicitly listing `__dict__` in `__slots__` adds a dictionary alongside declared slots. The ID still lives in its own slot.

In [6]:
class P05Record:
    __slots__ = ('record_id', '__dict__')
    def __init__(self, record_id): self.record_id = record_id

r = P05Record('R-1')
r.owner = 'Lee'
r.priority = 2
assert r.record_id == 'R-1'
assert vars(r) == {'owner': 'Lee', 'priority': 2}
assert 'record_id' not in vars(r)
del r.priority
assert vars(r) == {'owner': 'Lee'}
print('05:', r.record_id, vars(r))

05: R-1 {'owner': 'Lee'}


**Further practice:** Write a version without `__dict__`; show that adding `owner` raises `AttributeError`.

## Problem 06: Store a property in its backing slot

**Challenge:** Validate `name` through a property, store it in `_name`, and keep `age` slotted. Demonstrate failed validation and absence of a dictionary.

**Before revealing the solution:** predict which fields are slotted, which enter an instance dictionary, and which operations should raise errors, wherever applicable.

### Worked solution and reasoning

A property lives on the class and uses a descriptor; the storage name to declare is `_name`, not the property name `name`. Slot declarations do not themselves validate values.

In [7]:
class P06Person:
    __slots__ = ('_name', 'age')
    def __init__(self, name, age):
        self.name = name
        self.age = age
    @property
    def name(self): return self._name
    @name.setter
    def name(self, value):
        if not isinstance(value, str) or not value.strip():
            raise ValueError('nonempty name required')
        self._name = value.strip()

p = P06Person('  Alex  ', 30)
assert (p.name, p.age) == ('Alex', 30)
expect_raises(ValueError, setattr, p, 'name', '  ')
assert p.name == 'Alex' and not hasattr(p, '__dict__')
assert isinstance(P06Person.__dict__['name'], property)
print('06:', p.name, p.age)

06: Alex 30


**Further practice:** Add a property validator for `age`, backed by `_age`, and adapt the slots.

## Problem 07: Avoid same-class slot/property name collisions

**Challenge:** Demonstrate that declaring `name` both as a slot and a property in the **same class** fails at class creation; then show the repaired class.

**Before revealing the solution:** predict which fields are slotted, which enter an instance dictionary, and which operations should raise errors, wherever applicable.

### Worked solution and reasoning

Slot descriptors and class-level properties need the same namespace entry if given the same name. They conflict, so the property should use a distinct backing slot.

In [8]:
bad_definition = """
class BadCollision:
    __slots__ = ('name',)
    @property
    def name(self):
        return 'computed'
"""
expect_raises(ValueError, exec, bad_definition, {})

class P07Correct:
    __slots__ = ('_name',)
    def __init__(self, name): self._name = name
    @property
    def name(self): return self._name

obj = P07Correct('fixed')
assert obj.name == 'fixed'
expect_raises(AttributeError, setattr, obj, 'name', 'wrong')
print('07: collision rejected; backing slot works')

07: collision rejected; backing slot works


**Further practice:** Try an explicit setter and test that it updates `_name`.

## Problem 08: Inherit properties without re-declaring backing slots

**Challenge:** Create an employee with a read-only `name` property and a slotted engineer descendant with a new `language` field.

**Before revealing the solution:** predict which fields are slotted, which enter an instance dictionary, and which operations should raise errors, wherever applicable.

### Worked solution and reasoning

Inherited getters can access inherited backing storage. The descendant must declare only its additional slot; re-declaring `_name` may hide the ancestor slot descriptor.

In [9]:
class P08Employee:
    __slots__ = ('_name',)
    def __init__(self, name): self._name = name
    @property
    def name(self): return self._name

class P08Engineer(P08Employee):
    __slots__ = ('language',)
    def __init__(self, name, language):
        super().__init__(name)
        self.language = language
    def summary(self): return f'{self.name}: {self.language}'

e = P08Engineer('Morgan', 'Python')
assert e.summary() == 'Morgan: Python'
expect_raises(AttributeError, setattr, e, 'name', 'Other')
expect_raises(AttributeError, setattr, e, 'extra', 1)
assert not hasattr(e, '__dict__')
print('08:', e.summary())

08: Morgan: Python


**Further practice:** Allow a validated public name setter in the base and confirm the descendant can use it.

## Problem 09: Understand data-descriptor priority

**Challenge:** Put a writable property `status` in a slotted parent, and let an unslotted child get a dictionary. Insert a conflicting `status` dictionary key by hand; test which value wins.

**Before revealing the solution:** predict which fields are slotted, which enter an instance dictionary, and which operations should raise errors, wherever applicable.

### Worked solution and reasoning

Properties are data descriptors, so their access takes priority over same-named dictionary entries. A write through the property updates its backing slot rather than the manually inserted dictionary value.

In [10]:
class P09Base:
    __slots__ = ('_status',)
    def __init__(self, status): self.status = status
    @property
    def status(self): return self._status
    @status.setter
    def status(self, value): self._status = value.upper()

class P09Child(P09Base): pass

obj = P09Child('open')
obj.__dict__['status'] = 'dictionary copy'
assert obj.status == 'OPEN'
obj.status = 'closed'
assert obj.status == 'CLOSED'
assert obj.__dict__['status'] == 'dictionary copy'
print('09: descriptor:', obj.status, '| dictionary:', vars(obj)['status'])

09: descriptor: CLOSED | dictionary: dictionary copy


**Further practice:** Compare the precedence of a normal class variable with a matching instance-dictionary key.

## Problem 10: Why duplicate inherited slot names are dangerous

**Challenge:** Intentionally duplicate a base slot in its child; retrieve each descriptor independently and show that the parent descriptor can remain uninitialized. Fix it.

**Before revealing the solution:** predict which fields are slotted, which enter an instance dictionary, and which operations should raise errors, wherever applicable.

### Worked solution and reasoning

Re-declaring a parent slot in a descendant creates a separate descriptor and hides the inherited descriptor. This is a deliberate anti-pattern demonstration; do not copy it into production code.

In [11]:
class P10Base:
    __slots__ = ('code',)

class P10Bad(P10Base):
    __slots__ = ('code',)  # intentionally wrong

bad = P10Bad()
bad.code = 'child'
base_slot = P10Base.__dict__['code']
child_slot = P10Bad.__dict__['code']
assert base_slot is not child_slot
assert child_slot.__get__(bad, P10Bad) == 'child'
expect_raises(AttributeError, base_slot.__get__, bad, P10Bad)

class P10Good(P10Base):
    __slots__ = ('level',)
    def __init__(self, code, level):
        self.code = code
        self.level = level

good = P10Good('parent', 3)
assert (good.code, good.level) == ('parent', 3)
assert not hasattr(good, '__dict__')
print('10: duplicate slot diagnosed and repaired')

10: duplicate slot diagnosed and repaired


**Further practice:** Write a class-creation check that compares a child slot declaration with inherited slot names.

## Problem 11: Class constants versus slot-backed defaults

**Challenge:** Demonstrate that defining a same-named class variable and slot fails; then implement a shared constant and an instance-specific slot default.

**Before revealing the solution:** predict which fields are slotted, which enter an instance dictionary, and which operations should raise errors, wherever applicable.

### Worked solution and reasoning

A class constant and a member descriptor cannot both occupy the same class-namespace key. Use a distinct constant name or initialize the slot in `__init__`.

In [12]:
bad = """
class InvalidDefault:
    __slots__ = ('kind',)
    kind = 'member'
"""
expect_raises(ValueError, exec, bad, {})

class P11Constant:
    __slots__ = ('name',)
    KIND = 'member'
    def __init__(self, name): self.name = name

class P11Instance:
    __slots__ = ('kind',)
    def __init__(self, kind='member'): self.kind = kind

a, b = P11Constant('Rae'), P11Instance()
assert a.KIND == b.kind == 'member'
assert not hasattr(a, '__dict__') and not hasattr(b, '__dict__')
print('11: class constant separated from slot storage')

11: class constant separated from slot storage


**Further practice:** Make the class constant read-only by convention and explain why that is not a security guarantee.

## Problem 12: Allow weak references without a dictionary

**Challenge:** Compare a slotted object that rejects `weakref.ref()` with a slotted object whose slots include `__weakref__`.

**Before revealing the solution:** predict which fields are slotted, which enter an instance dictionary, and which operations should raise errors, wherever applicable.

### Worked solution and reasoning

Slot-only classes generally need the special `__weakref__` slot (or inherited weak-reference support) to permit weak references. It does not imply an instance dictionary. Do not assert immediate collection timing.

In [13]:
import weakref
class P12Blocked:
    __slots__ = ('value',)
    def __init__(self, value): self.value = value
class P12Ready:
    __slots__ = ('value', '__weakref__')
    def __init__(self, value): self.value = value

expect_raises(TypeError, weakref.ref, P12Blocked(1))
obj = P12Ready(2)
ref = weakref.ref(obj)
assert ref() is obj and obj.value == 2
assert not hasattr(obj, '__dict__')
print('12: weak references explicitly enabled')

12: weak references explicitly enabled


**Further practice:** Subclass `P12Ready` with `__slots__ = ()` and verify inherited weak-reference capability.

## Problem 13: Integrate slotted dataclasses with inherited fields

**Challenge:** Use `@dataclass(slots=True)` for base and child; enumerate fields correctly and ensure instances reject unplanned fields.

**Before revealing the solution:** predict which fields are slotted, which enter an instance dictionary, and which operations should raise errors, wherever applicable.

### Worked solution and reasoning

`dataclasses.fields()` describes dataclass fields independently of class-layout specifics. Exact `__slots__` membership across inheritance changed in Python 3.11, so do not treat `__slots__` as a universal field API. Exercise requires Python 3.10+.

In [14]:
from dataclasses import dataclass, fields

@dataclass(slots=True)
class P13Asset:
    code: str

@dataclass(slots=True)
class P13Device(P13Asset):
    serial: str

obj = P13Device('A-1', 'S-8')
assert (obj.code, obj.serial) == ('A-1', 'S-8')
assert [f.name for f in fields(P13Device)] == ['code', 'serial']
assert not hasattr(obj, '__dict__')
expect_raises(AttributeError, setattr, obj, 'extra', 10)
print('13:', obj, [f.name for f in fields(obj)])

13: P13Device(code='A-1', serial='S-8') ['code', 'serial']


**Further practice:** Add a defaulted child field after required fields; do not assume a leaf class declares inherited fields in its own slots.

## Problem 14: Slots do not enforce types or immutability

**Challenge:** Create a stock item with a mutable SKU slot and validated `quantity` property. Reject Boolean values, negative values and strings; preserve state on failed updates.

**Before revealing the solution:** predict which fields are slotted, which enter an instance dictionary, and which operations should raise errors, wherever applicable.

### Worked solution and reasoning

Slots control ordinary attribute names, not types or value constraints. Python `bool` is a subclass of `int`, requiring an explicit rejection when an integer count is expected.

In [15]:
class P14Stock:
    __slots__ = ('sku', '_quantity')
    def __init__(self, sku, quantity):
        self.sku = sku
        self.quantity = quantity
    @property
    def quantity(self): return self._quantity
    @quantity.setter
    def quantity(self, value):
        if isinstance(value, bool) or not isinstance(value, int) or value < 0:
            raise ValueError('quantity must be nonnegative integer')
        self._quantity = value

s = P14Stock('A', 5)
s.sku = 'B'
s.quantity = 8
for invalid in (-1, True, '10'):
    expect_raises(ValueError, setattr, s, 'quantity', invalid)
assert s.quantity == 8 and s.sku == 'B'
expect_raises(AttributeError, setattr, s, 'color', 'blue')
print('14: validated quantity:', s.quantity)

14: validated quantity: 8


**Further practice:** Make SKU read-only through a public property with a separately slotted backing field.

## Problem 15: Unset slots, deletion and reassignment

**Challenge:** Create a two-slot class that initializes only one slot. Read the unset slot, set it, delete the first and assign it again.

**Before revealing the solution:** predict which fields are slotted, which enter an instance dictionary, and which operations should raise errors, wherever applicable.

### Worked solution and reasoning

Unlike dictionary keys, slot descriptors exist as class members even when their per-instance storage is unset. Reading an unset slot raises `AttributeError`; deleting a set slot unsets it.

In [16]:
class P15Pair:
    __slots__ = ('left', 'right')
    def __init__(self, left): self.left = left

p = P15Pair(1)
expect_raises(AttributeError, getattr, p, 'right')
p.right = 2
assert p.right == 2
del p.left
expect_raises(AttributeError, getattr, p, 'left')
p.left = 3
assert (p.left, p.right) == (3, 2)
expect_raises(AttributeError, setattr, p, 'middle', 0)
print('15: unset versus undeclared attributes demonstrated')

15: unset versus undeclared attributes demonstrated


**Further practice:** Ensure all declared slots are initialized in `__init__` and compare the resulting API.

## Problem 16: Introspect inherited slot declarations

**Challenge:** Write a common-case helper that reads the slot names declared on every class in the MRO, excluding `__dict__` and `__weakref__`.

**Before revealing the solution:** predict which fields are slotted, which enter an instance dictionary, and which operations should raise errors, wherever applicable.

### Worked solution and reasoning

Reading only the leaf class’s `__slots__` misses inherited declarations. Inspect each class’s *own* dictionary and handle a single string as one slot. This is an educational helper, not a general serializer: unusual slot iterables, name mangling and descriptors need extra work.

In [17]:
def list_declared_slots(cls):
    result = []
    for owner in reversed(cls.__mro__):
        own = owner.__dict__.get('__slots__', ())
        if isinstance(own, str):
            own = (own,)
        for slot in own:
            if slot not in ('__dict__', '__weakref__'):
                result.append(slot)
    return tuple(result)

class P16Base:
    __slots__ = ('name',)
class P16Child(P16Base):
    __slots__ = ('score', '__dict__')

obj = P16Child()
obj.name, obj.score, obj.note = 'Ali', 98, 'dynamic'
assert list_declared_slots(P16Child) == ('name', 'score')
assert vars(obj) == {'note': 'dynamic'}
print('16:', list_declared_slots(P16Child), vars(obj))

16: ('name', 'score') {'note': 'dynamic'}


**Further practice:** Test a class declaring `__slots__ = "only_one"` and confirm it produces one slot name, not individual characters.

## Problem 17: Correctly clone both slotted and dictionary fields

**Challenge:** A child combines an inherited ID slot, its own score slot and dynamic notes. Clone it without losing slot data.

**Before revealing the solution:** predict which fields are slotted, which enter an instance dictionary, and which operations should raise errors, wherever applicable.

### Worked solution and reasoning

`vars(original)` sees only dictionary fields and misses inherited or local slots. For a known model, a constructor-based clone plus explicit dictionary copying is transparent and maintainable. This version performs a shallow copy.

In [18]:
class P17Base:
    __slots__ = ('identifier',)
    def __init__(self, identifier): self.identifier = identifier

class P17Child(P17Base):
    __slots__ = ('score', '__dict__')
    def __init__(self, identifier, score):
        super().__init__(identifier)
        self.score = score
    def clone(self):
        result = type(self)(self.identifier, self.score)
        result.__dict__.update(self.__dict__)
        return result

original = P17Child('C-3', 72)
original.comment = 'initial'
clone = original.clone()
assert clone is not original
assert (clone.identifier, clone.score, clone.comment) == ('C-3', 72, 'initial')
clone.score = 99
clone.comment = 'updated'
assert original.score == 72 and original.comment == 'initial'
print('17: inherited slot, child slot, dictionary preserved')

17: inherited slot, child slot, dictionary preserved


**Further practice:** Add a mutable `tags` list, test shallow aliasing and design a deep-copy alternative.

## Problem 18: Deep-copy nested mutable state

**Challenge:** Model a slotted folder holding a list of dictionaries, deep-copy it, then mutate a nested list in the copy.

**Before revealing the solution:** predict which fields are slotted, which enter an instance dictionary, and which operations should raise errors, wherever applicable.

### Worked solution and reasoning

The slots store references and do not make nested containers immutable. `copy.deepcopy` handles ordinary slot-backed objects and nested Python collections; custom resources may require special copy behavior.

In [19]:
import copy
class P18Folder:
    __slots__ = ('name', 'entries')
    def __init__(self, name, entries):
        self.name, self.entries = name, entries

source = P18Folder('work', [{'tasks': ['draft']}])
other = copy.deepcopy(source)
other.entries[0]['tasks'].append('review')
assert other is not source
assert other.entries is not source.entries
assert source.entries[0]['tasks'] == ['draft']
assert other.entries[0]['tasks'] == ['draft', 'review']
assert not hasattr(other, '__dict__')
print('18: nested states independent after deep copy')

18: nested states independent after deep copy


**Further practice:** Identify resources that should *not* be deep-copied, such as file handles and locks.

## Problem 19: Serialize inherited slotted fields with an explicit JSON schema

**Challenge:** Round-trip a slotted student with an inherited name and child school through JSON; reject missing fields without using `vars`.

**Before revealing the solution:** predict which fields are slotted, which enter an instance dictionary, and which operations should raise errors, wherever applicable.

### Worked solution and reasoning

A public schema is more robust than guessing storage from `vars` or slot declarations. Validate required JSON keys before constructing an object; consider schema versioning for lasting formats.

In [20]:
import json
class P19Person:
    __slots__ = ('name',)
    def __init__(self, name): self.name = name
class P19Student(P19Person):
    __slots__ = ('school',)
    def __init__(self, name, school):
        super().__init__(name)
        self.school = school
    def to_record(self):
        return {'name': self.name, 'school': self.school}
    @classmethod
    def from_record(cls, value):
        if not isinstance(value, dict) or not {'name', 'school'} <= value.keys():
            raise ValueError('name and school are required')
        return cls(value['name'], value['school'])

original = P19Student('Kim', 'Example College')
text = json.dumps(original.to_record(), sort_keys=True)
restored = P19Student.from_record(json.loads(text))
assert restored.to_record() == original.to_record()
expect_raises(ValueError, P19Student.from_record, {'name': 'Kim'})
expect_raises(TypeError, vars, original)
print('19:', text)

19: {"name": "Kim", "school": "Example College"}


**Further practice:** Add schema version 2, and write a migration from a version 1 key named `institution`.

## Problem 20: Pickle trusted mixed-layout objects

**Challenge:** Pickle and unpickle a top-level class with a slot-backed identifier and dictionary-backed tag; check both survive.

**Before revealing the solution:** predict which fields are slotted, which enter an instance dictionary, and which operations should raise errors, wherever applicable.

### Worked solution and reasoning

Python pickle generally supports ordinary slotted classes, including mixed slot/dictionary layouts. **Never unpickle untrusted data**: loading pickle can execute arbitrary code. Use explicit JSON schemas across trust boundaries.

In [21]:
import pickle
class P20Artifact:
    __slots__ = ('identifier', '__dict__')
    def __init__(self, identifier): self.identifier = identifier

artifact = P20Artifact('A-1')
artifact.tag = 'internal'
payload = pickle.dumps(artifact, protocol=pickle.HIGHEST_PROTOCOL)
restored = pickle.loads(payload)  # trusted bytes produced locally above
assert restored is not artifact
assert restored.identifier == 'A-1'
assert vars(restored) == {'tag': 'internal'}
assert 'identifier' not in vars(restored)
print('20: trusted local pickle preserved slot and dictionary')

20: trusted local pickle preserved slot and dictionary


**Further practice:** Implement `to_record` / `from_record` and JSON instead for data received from outside your application.

## Problem 21: Measure instance memory without misleading conclusions

**Challenge:** Compare memory for 10,000 ordinary versus slotted instances; show both `sys.getsizeof` and separately measured `tracemalloc` allocations.

**Before revealing the solution:** predict which fields are slotted, which enter an instance dictionary, and which operations should raise errors, wherever applicable.

### Worked solution and reasoning

`sys.getsizeof` is shallow and does not account for the complete object graph. `tracemalloc` sees traced Python allocations, not every external allocation. Do not assert a universal byte difference or speedup: allocator, version and runtime matter.

In [22]:
import gc
import sys
import tracemalloc
class P21Plain:
    def __init__(self, x, y): self.x, self.y = x, y
class P21Slotted:
    __slots__ = ('x', 'y')
    def __init__(self, x, y): self.x, self.y = x, y

def measured_allocation(factory, count=10_000):
    gc.collect()
    tracemalloc.start()
    tracemalloc.reset_peak()
    try:
        instances = [factory(1, 2) for _ in range(count)]
        current, peak = tracemalloc.get_traced_memory()
        assert len(instances) == count
        return current, peak
    finally:
        tracemalloc.stop()

ordinary = measured_allocation(P21Plain)
slotted = measured_allocation(P21Slotted)
assert all(v >= 0 for v in ordinary + slotted)
print('21: ordinary current/peak:', ordinary)
print('21: slotted current/peak:', slotted)
print('21: shallow instance bytes:', sys.getsizeof(P21Plain(1, 2)), sys.getsizeof(P21Slotted(1, 2)))

21: ordinary current/peak: (968480, 968544)
21: slotted current/peak: (565120, 565230)
21: shallow instance bytes: 48 48


**Further practice:** Reverse measurement order and repeat several times. Add an unslotted descendant of the slotted class.

## Problem 22: Benchmark attribute reads fairly

**Challenge:** Benchmark a plain and a slotted instance using `timeit.repeat`, avoiding object creation in the timed statement. Compare medians without asserting a winner.

**Before revealing the solution:** predict which fields are slotted, which enter an instance dictionary, and which operations should raise errors, wherever applicable.

### Worked solution and reasoning

Interpreter optimizations, hardware, environment and workload affect timing. Measure rather than assuming a slotted class has a universal access-speed advantage. Benchmarks should be reproducible and accompanied by correctness tests.

In [23]:
import statistics
import timeit
class P22Plain:
    def __init__(self, value): self.value = value
class P22Slotted:
    __slots__ = ('value',)
    def __init__(self, value): self.value = value

plain, slot = P22Plain(5), P22Slotted(5)
assert plain.value == slot.value == 5

def median_access(obj):
    return statistics.median(timeit.Timer('obj.value', globals={'obj': obj}).repeat(repeat=3, number=100_000))

plain_time, slot_time = median_access(plain), median_access(slot)
assert plain_time >= 0 and slot_time >= 0
print(f'22: 100,000 reads median of 3: ordinary={plain_time:.6f}s, slots={slot_time:.6f}s')

22: 100,000 reads median of 3: ordinary=0.001857s, slots=0.002074s


**Further practice:** Benchmark attribute writes and properties independently, keeping correctness tests outside the timed statement.

## Problem 23: Validate finite numeric values across an inheritance chain

**Challenge:** Create a slotted profile with an identifier and a child with a score property accepting finite real values from 0 to 100, excluding booleans. Failed writes must preserve old state.

**Before revealing the solution:** predict which fields are slotted, which enter an instance dictionary, and which operations should raise errors, wherever applicable.

### Worked solution and reasoning

Keep storage names in the slots and business rules in the setter. Since `nan` fails ordinary comparisons unpredictably, explicitly test `math.isfinite` before storing. Validate before mutating to maintain the invariant.

In [24]:
import math
from numbers import Real
class P23Profile:
    __slots__ = ('identifier',)
    def __init__(self, identifier): self.identifier = identifier
class P23Graded(P23Profile):
    __slots__ = ('_score',)
    def __init__(self, identifier, score):
        super().__init__(identifier)
        self.score = score
    @property
    def score(self): return self._score
    @score.setter
    def score(self, value):
        if (isinstance(value, bool) or not isinstance(value, Real)
                or not math.isfinite(value) or not 0 <= value <= 100):
            raise ValueError('score must be finite and between 0 and 100')
        self._score = value

obj = P23Graded('ID-1', 81.5)
for invalid in (-1, 101, True, '97', float('nan'), float('inf')):
    expect_raises(ValueError, setattr, obj, 'score', invalid)
assert obj.score == 81.5
expect_raises(AttributeError, setattr, obj, 'extra', 'wrong')
assert not hasattr(obj, '__dict__')
print('23: invalid scores rejected; previous score preserved')

23: invalid scores rejected; previous score preserved


**Further practice:** Add a derived read-only `passed` property without allocating a separate `passed` slot.

## Problem 24: Capstone: fixed-layout account with weakrefs, permissions and JSON

**Challenge:** Build a slotted `Account` base and slotted `Member` descendant. Support weakrefs, a read-only public identifier, a validated display name, defensively copied permissions, a read-only permission snapshot, grant, JSON-ready records and a round trip.

**Before revealing the solution:** predict which fields are slotted, which enter an instance dictionary, and which operations should raise errors, wherever applicable.

### Worked solution and reasoning

This integrates parent/child slots, inherited descriptor access, backing fields, `__weakref__`, explicit schema, defensive copying and exception tests. Read-only public APIs and underscore attributes are design conventions, not security boundaries; do not expose the mutable permissions set itself.

In [25]:
import json
import weakref

class P24Account:
    __slots__ = ('_account_id', '__weakref__')
    def __init__(self, account_id):
        if not isinstance(account_id, str) or not account_id.strip():
            raise ValueError('account_id required')
        self._account_id = account_id.strip()
    @property
    def account_id(self): return self._account_id

class P24Member(P24Account):
    __slots__ = ('_display_name', '_permissions')
    def __init__(self, account_id, display_name, permissions=()):
        super().__init__(account_id)
        self.display_name = display_name
        self._permissions = self._check_permissions(permissions)
    @staticmethod
    def _check_permissions(values):
        if isinstance(values, (str, bytes)):
            raise ValueError('permissions must be iterable, not text')
        try:
            result = set(values)
        except (TypeError, ValueError) as exc:
            raise ValueError('invalid permissions iterable') from exc
        if any(not isinstance(v, str) or not v.strip() for v in result):
            raise ValueError('permissions must be nonempty strings')
        return result
    @property
    def display_name(self): return self._display_name
    @display_name.setter
    def display_name(self, value):
        if not isinstance(value, str) or not value.strip():
            raise ValueError('display name required')
        self._display_name = value.strip()
    @property
    def permissions(self): return frozenset(self._permissions)
    def grant(self, permission):
        self._permissions.update(self._check_permissions((permission,)))
    def to_record(self):
        return {'account_id': self.account_id,
                'display_name': self.display_name,
                'permissions': sorted(self._permissions)}
    @classmethod
    def from_record(cls, record):
        if not isinstance(record, dict):
            raise ValueError('record must be a dictionary')
        try:
            return cls(record['account_id'], record['display_name'], record['permissions'])
        except KeyError as exc:
            raise ValueError(f'missing key: {exc.args[0]}') from exc

external = ['read']
member = P24Member(' A-1 ', '  Casey ', external)
external.append('admin')
assert (member.account_id, member.display_name) == ('A-1', 'Casey')
assert member.permissions == frozenset({'read'})
assert weakref.ref(member)() is member
assert not hasattr(member, '__dict__')
expect_raises(AttributeError, setattr, member, 'account_id', 'changed')
expect_raises(AttributeError, setattr, member, 'random', 10)
expect_raises(ValueError, setattr, member, 'display_name', ' ')
expect_raises(ValueError, member.grant, '')
member.grant('write')
assert member.permissions == frozenset({'read', 'write'})
encoded = json.dumps(member.to_record(), sort_keys=True)
restored = P24Member.from_record(json.loads(encoded))
assert restored.to_record() == member.to_record()
expect_raises(ValueError, P24Member.from_record, {'account_id': 'A-1'})
print('24 CAPSTONE:', encoded)

24 CAPSTONE: {"account_id": "A-1", "display_name": "Casey", "permissions": ["read", "write"]}


**Further practice:** Implement `revoke()` and test absent permissions. Add schema versioning and round-trip tests for a future new slot.

---
## Debugging checklist and key distinctions

| Question | Correct principle |
|---|---|
| Slotted parent, unslotted child? | Child normally gets an instance dictionary; parent slots remain. |
| Ordinary parent, slotted child? | Child still inherits the parent dictionary. |
| Child adds no new attributes? | Declare `__slots__ = ()` to stay fixed-layout if bases permit. |
| Child adds slots? | Declare only the **new** names; do not duplicate ancestor slots. |
| Property named `name`? | Store data in a distinct declared backing slot such as `_name`. |
| Does `vars(obj)` find slots? | No. It accesses `obj.__dict__` when present. |
| Are slots immutable or type-safe? | No; add explicit properties/validation as needed. |
| Are slots always faster and smaller? | No universal promise; measure with representative workloads. |
| How to serialize? | Prefer explicit, stable public schemas; never unpickle untrusted data. |

**Study workflow:** Cover each solution; write your own implementation in a new cell; run the assertions; compare against the worked solution; finally complete each further-practice task. Scope is deliberately **single inheritance**, matching the original lesson.

In [26]:
print('SUCCESS: all 24 problems and their automated assertions completed.')

SUCCESS: all 24 problems and their automated assertions completed.
